## PharmaNODE — Generic reader/collate + FiLM training\n
\n
This notebook is for **new users** training the FiLM model on data generated from **their own PK model**.\n
\n
It shows how to:\n
- map your CSV schema to the **FiLM internal data contract**\n
- write/adjust the reader (`extract_user_film_generic`)\n
- write/adjust the collate (`collate_fn_generic_film`)\n
- run end-to-end training using the same FiLM batch path as `run_models.py`\n
\n
### Important shape constraint\n
The encoder has a `static_encoder = Linear(3, ...)` (hard-coded), so **`static` must have 3 values per patient/visit**.\n
Default in the generic reader is `static = [dose_norm, 0, 0]`.

In [2]:
import os
from types import SimpleNamespace

import numpy as np
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.distributions.normal import Normal

import sys
# 1. Add the parent directory to the Python path
sys.path.append(os.path.abspath('..'))

import lib.utils as utils
from lib.create_latent_ode_model import create_LatentODE_model

from lib.read_user_film_generic import (
    GenericFilmSchema,
    extract_user_film_generic,
    GenericFilmDataset,
    collate_fn_generic_film,
)

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('device:', device)

device: cpu


### 1) Point to your CSVs\n
\n
If you used `01_generate_film_dataset_generic.ipynb`, your files are under `exp_run_all/<exp_name>/`.

In [3]:
exp_name = 'exp_film_generic'  # must match Notebook 1 `cfg.exp_name`
train_csv = os.path.join('exp_run_all', exp_name, 'virtual_cohort_film_train.csv')
test_csv = os.path.join('exp_run_all', exp_name, 'virtual_cohort_film_test.csv')

assert os.path.exists(train_csv), f"Missing {train_csv}"
assert os.path.exists(test_csv), f"Missing {test_csv}"
print('Train:', train_csv)
print('Test :', test_csv)

Train: exp_run_all/exp_film_generic/virtual_cohort_film_train.csv
Test : exp_run_all/exp_film_generic/virtual_cohort_film_test.csv


### 2) Schema mapping and encoder sparse times\n
\n
- If you used the generator notebook defaults, keep the schema as-is.\n
- If your CSV uses different names, update these mappings.\n
- Current FiLM collate expects **exactly 3** encoder timepoints.

In [4]:
schema = GenericFilmSchema(
    id_col='ID',
    visit_col='VISIT',
    time_col='TIME',
    dv_col='DV',
    amt_col='AMT',
    mdv_col='mdv',
)

encoder_times_h = (0.0, 1.0, 3.0)
print('encoder_times_h:', encoder_times_h)

encoder_times_h: (0.0, 1.0, 3.0)


### 3) Load data with the generic reader\n
\n
`extract_user_film_generic` builds the internal `data_dict` needed for FiLM training.

In [5]:
data_dict_train, scaler = extract_user_film_generic(
    [train_csv],
    schema=schema,
    encoder_times_h=encoder_times_h,
    apply_boxcox=True,
)
data_dict_test, _ = extract_user_film_generic(
    [test_csv],
    schema=schema,
    encoder_times_h=encoder_times_h,
    apply_boxcox=True,
)

print('Train patients:', len(data_dict_train))
print('Test patients :', len(data_dict_test))
print('Scaler:', scaler)

Train patients: 160
Test patients : 40
Scaler: {'max_out': 0.228392481803894, 'best_lambda': np.float64(0.07860099913438592)}


### 4) Build DataLoaders and validate FiLM batch shapes\n
\n
This is the exact pre-processing step that feeds `model.compute_film_losses(...)`.

In [6]:
batch_size = 32

train_ds = GenericFilmDataset(data_dict_train)
test_ds = GenericFilmDataset(data_dict_test)

train_dl = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=lambda b: collate_fn_generic_film(b, device, encoder_times_h=encoder_times_h),
)
test_dl = DataLoader(
    test_ds,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=lambda b: collate_fn_generic_film(b, device, encoder_times_h=encoder_times_h),
)

data_obj = {
    'input_dim': 1,
    'train_dataloader': utils.inf_generator(train_dl),
    'test_dataloader': utils.inf_generator(test_dl),
    'n_train_batches': len(train_dl),
    'n_test_batches': len(test_dl),
    'max_out': {'max_out': np.array(scaler['max_out']), 'best_lambda': np.array(scaler['best_lambda'])},
}

batch0 = utils.get_next_batch_film(data_obj['train_dataloader'])
for k, v in batch0.items():
    if torch.is_tensor(v):
        print(k, tuple(v.shape), v.dtype)
    else:
        print(k, type(v))

observed_data_v1 (32, 3, 1) torch.float32
observed_tp_v1 (3,) torch.float32
dose_v1 (32,) torch.float32
auc_red_v1 (32,) torch.float32
others_v1 (32, 6) torch.float32
static_v1 (32, 3) torch.float32
data_to_predict_v1 (32, 11, 1) torch.float32
tp_to_predict_v1 (11,) torch.float32
observed_data_v2 (32, 3, 1) torch.float32
observed_tp_v2 (3,) torch.float32
dose_v2 (32,) torch.float32
auc_red_v2 (32,) torch.float32
others_v2 (32, 6) torch.float32
static_v2 (32, 3) torch.float32
data_to_predict_v2 (32, 11, 1) torch.float32
tp_to_predict_v2 (11,) torch.float32
delta_t (32, 1) torch.float32
t_v1 (32, 1) torch.float32
patient_ids <class 'list'>


### 5) Create the FiLM model\n
\n
We construct an `args` object with the fields used by `create_LatentODE_model`.

In [7]:
args = SimpleNamespace(
    dataset='PK_Generic',
    seed=15,
    latent_ode=True,
    classic_rnn=False,
    ode_rnn=False,
    rnn_vae=False,
    use_film=True,
    latents=6,
    rec_dims=20,
    rec_layers=1,
    gen_layers=1,
    units=100,
    gru_units=100,
    z0_encoder='odernn',
    poisson=False,
    classif=False,
    linear_classif=False,
    use_gmm=False,
    use_gmm_v=False,
    use_flow=False,
    n_components=4,
)

input_dim = 1
obsrv_std = torch.Tensor([0.01]).to(device)
z0_prior = Normal(torch.Tensor([0.0]).to(device), torch.Tensor([1.0]).to(device))

model = create_LatentODE_model(args, input_dim, z0_prior, obsrv_std, device)
model.to(device)
print('Model ready.')

Model ready.


### 6) Train end-to-end (FiLM)\n
\n
This uses the same FiLM batch path (`utils.get_next_batch_film`) and loss (`model.compute_film_losses`) as the FiLM training code.

In [9]:
lr = 1e-3
niters = 6000

optimizer = optim.Adamax(model.parameters(), lr=lr)

for itr in range(1, niters + 1):
    optimizer.zero_grad()
    batch_dict = utils.get_next_batch_film(data_obj['train_dataloader'])
    train_res = model.compute_film_losses(
        batch_dict,
        n_traj_samples=3,
        kl_coef=0.0,
        max_out=data_obj['max_out'],
    )
    train_res['loss'].backward()
    optimizer.step()

    if itr % 100 == 0:
        with torch.no_grad():
            test_batch = utils.get_next_batch_film(data_obj['test_dataloader'])
            test_res = model.compute_film_losses(
                test_batch,
                n_traj_samples=3,
                kl_coef=0.0,
                max_out=data_obj['max_out'],
            )
        print(
            f"itr {itr:04d} | train_loss={train_res['loss'].item():.4f} | "
            f"train_mse={train_res.get('mse', torch.tensor(float('nan'))).item():.4f} | "
            f"test_loss={test_res['loss'].item():.4f}"
        )

itr 0020 | train_loss=2383.8955 | train_mse=0.2446 | test_loss=2595.7656
itr 0040 | train_loss=2062.6377 | train_mse=0.2101 | test_loss=2462.6248
itr 0060 | train_loss=2114.5754 | train_mse=0.2185 | test_loss=2070.5496
itr 0080 | train_loss=1683.6765 | train_mse=0.2157 | test_loss=1909.9685
itr 0100 | train_loss=1590.1213 | train_mse=0.2093 | test_loss=1845.7839
itr 0120 | train_loss=1272.3395 | train_mse=0.1601 | test_loss=1819.5253
itr 0140 | train_loss=1416.6520 | train_mse=0.1903 | test_loss=1633.2720
itr 0160 | train_loss=1277.9858 | train_mse=0.1593 | test_loss=1588.5092
itr 0180 | train_loss=1122.6960 | train_mse=0.1417 | test_loss=1495.8274
itr 0200 | train_loss=1076.2719 | train_mse=0.1412 | test_loss=1410.3472
itr 0220 | train_loss=917.7235 | train_mse=0.1100 | test_loss=1299.5663
itr 0240 | train_loss=918.2922 | train_mse=0.1205 | test_loss=1192.4860
itr 0260 | train_loss=839.5784 | train_mse=0.0921 | test_loss=1171.9326
itr 0280 | train_loss=814.6631 | train_mse=0.1049 | te

### 7) Save checkpoint

In [10]:
os.makedirs('experiments', exist_ok=True)
ckpt_path = os.path.join('experiments', f"experiment_film_{exp_name}_generic.ckpt")
torch.save({'args': args, 'state_dict': model.state_dict()}, ckpt_path)
print('Saved:', ckpt_path)

Saved: experiments/experiment_film_exp_film_generic_generic.ckpt


### 8) Evaluate and Plot Performance

Now that the model is trained, let's visualize its extrapolation performance on a batch from the test set. 

We will use the `Visualizations` class, which includes a specific method for FiLM (`draw_all_plots_film`). This method automatically applies the inverse Box-Cox transformation and rescales the predictions back to their original concentration values so you can easily assess the fit.

In [12]:
import matplotlib.pyplot as plt
from lib.plotting import Visualizations

# 1. Initialize the visualizer
viz = Visualizations(device)

# 2. Set the model to evaluation mode
model.eval()

# 3. Grab a batch from the test dataloader
with torch.no_grad():
    test_batch = utils.get_next_batch_film(data_obj['test_dataloader'])

    # 4. Generate the FiLM specific extrapolation plots
    # We pass the max_out dictionary so the plotting function can unscale the data
    print("Generating Extrapolation Plots...")
    viz.draw_all_plots_film(
        batch_dict=test_batch,
        model=model,
        plot_name=f"eval_film_{exp_name}.png",
        experimentID=exp_name,
        save=True,  # Set to True if you want to save it to the plots/ directory
        scaler=data_obj['max_out']
    )
    
    # Ensure the plot is displayed inline in the notebook
    plt.show()

/Users/benjaminmaurel/Documents/PharmaNODE/lib/plotting.py:479: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show(block=False)


Generating Extrapolation Plots...


/var/folders/1j/drsjnlsx1fx7krl2q15mzjfh0000gn/T/ipykernel_38930/297091274.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
